# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I'm choosing **Lane 2: Refresh / Content Opportunity Scoring**.

The starter pipeline in this repo already builds a working example of this exact lane end-to-end 
(feature prep → baseline rule → trained model → ranked queue), which gives me a proven workflow to 
learn from and extend. It also produces a clear, measurable win: the baseline rule scores 
Precision@50 = 0.240, while a random forest reaches 0.740 on the same data — meaning a learned 
model finds roughly 3x more true positives in its top 50 picks than a hand-written rule. That's a 
concrete signal this direction has real room for an ML model to add value over a simple heuristic, 
which is exactly what I want to explore and improve over the next 7 weeks (e.g., using a proper 
future-window label instead of the current-window proxy label).

## 2. The question: decision, action, cost of a wrong call

**Question:** Which pages in a client's content inventory should be reviewed first for refresh, 
expansion, or protection, given limited reviewer time?

**Unit of analysis:** one content page (`content_id`), scored using its 90-day window of signals.

**Decision it improves:** which pages an SEO/content reviewer looks at first this week/month, out 
of potentially thousands of candidate pages.

**Who acts on it:** a content strategist or SEO reviewer with limited bandwidth (they can't review 
every page every week).

**Output:** a ranked queue of pages, each with a score and a reason code explaining why it was 
flagged (e.g. `stale_visible_page`, `declining_with_demand`).

**Cost of a wrong recommendation:**
- **False positive** (flagging a healthy page as needing review): wastes reviewer time on a page 
  that didn't need attention, while a genuinely declining page waits longer.
- **False negative** (missing a page that's actually declining): the page keeps losing visibility/
  traffic silently until someone notices it much later — lost organic traffic compounds over time.

**Why data/ML can help:** with thousands of pages and only a handful of reviewer-hours, someone has 
to rank them. A hand-written rule (the baseline) only combines a few signals in a fixed way. A model 
can learn non-obvious combinations of signals (e.g. "position 8 + low CTR + old age" interacting 
differently than any one signal alone) — the starter pipeline's own baseline-vs-model comparison 
(Precision@50 0.240 vs 0.740) is direct evidence this isn't just "run a rule," there's real pattern 
to learn.

## 3. Quick look at the data

Loading `content_refresh_anonymized.csv` shows:

- **30,000 total pages** in the starter slice.
- **All 30,000 pages have impressions > 0** — meaning every page in this sample has at least some 
  search visibility to evaluate, so there's no dead weight in the data.
- **16,262 pages (54.2%) are currently in a "down" trend** — over half the inventory shows some 
  form of decline, which means there's a large, meaningful pool of candidates for a refresh-priority 
  model to rank — not just a handful of edge cases.
- The starter pipeline's own baseline vs. model comparison backs this up: Precision@50 goes from 
  **0.240 (hand-written baseline rule)** to **0.740 (random forest)** — about 3x more true positives 
  caught in the top 50 recommendations. With over half the pages showing decline, picking the 
  *right* 50 to review first clearly isn't trivial — and this gap shows a learned model can do it 
  meaningfully better than a fixed rule.

This combination — a large pool of declining pages plus clear evidence a model beats a simple rule — 
is why Lane 2 looks worth the next 7 weeks.

In [2]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Total pages:", len(df))
print("Pages with impressions > 0:", (df["impressions_90d"] > 0).sum())
print("Declining pages (trend_direction == 'down'):", (df["trend_direction"] == "down").sum())
print("Share declining:", round((df["trend_direction"] == "down").mean() * 100, 1), "%")

Total pages: 30000
Pages with impressions > 0: 30000
Declining pages (trend_direction == 'down'): 16262
Share declining: 54.2 %


## 4. Careful words: what I can and can't claim

**What I CAN claim:**
- Observed patterns: "pages with X signal combination were more likely to be in a declining trend 
  in this dataset."
- Directional evidence: "a learned ranking outperforms the hand-written baseline rule on this 
  starter slice, based on Precision@50."
- Decision-support: "this ranking helps a reviewer prioritize which pages to look at first — it 
  does not guarantee any single page will recover."

**What I will NEVER claim:**
- That I predicted or reverse-engineered Google's ranking algorithm.
- That refreshing a flagged page will cause it to recover (that needs a real experiment/causal 
  design, which this data can't provide).
- That the current `trend_direction == "down"` label is a perfect target — it's a proxy based on 
  the current window, not a true future outcome, so I'll treat early results as a starting point 
  to improve on with a future-window label later.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.